**Loading libraries and data checking**

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import re


# Read the CSV file into a DataFrame
try:
    df = pd.read_csv('phone_market_data.csv')
    print("CSV file loaded successfully.")
    print(f"Total data size: {len(df)} lines")

    # Display the first few rows of the DataFrame
    display(df.head())
except FileNotFoundError:
    print("❌'phone_market_data.csv' not found. Please ensure the file is in the correct directory.")

CSV file loaded successfully.
Total data size: 283 lines


,Name,Price,Date
0,Apple iPhone 13 Pro 128GB (Used),130000,2026-03-29
1,Xiaomi Redmi Note 11 SE 5G 6GB / 128GB (Used),21500,2026-03-29
2,Apple iPhone X 256GB (Used),47999,2026-03-29
3,Apple iPhone 13 Pro Max ZD/A (Used),140000,2026-03-29
4,Apple iPhone 13 128GB (Used),80000,2026-03-29


**Data cleaning and storage allocation**

In [2]:
# A function to get the GB value from the phone name
def extract_capacity(name):
    match = re.search(r'(\d+)\s*(GB|TB|gb|tb)', str(name))
    return match.group(0).upper() if match else "Standard"

# Create a new column 'Capacity'.
df['Capacity'] = df['Name'].apply(extract_capacity)

# The date is made in a way that pandas can understand.
df['Date'] = pd.to_datetime(df['Date'])

# Display the data
display(df.head())


,Name,Price,Date,Capacity
0,Apple iPhone 13 Pro 128GB (Used),130000,2026-03-29,128GB
1,Xiaomi Redmi Note 11 SE 5G 6GB / 128GB (Used),21500,2026-03-29,6GB
2,Apple iPhone X 256GB (Used),47999,2026-03-29,256GB
3,Apple iPhone 13 Pro Max ZD/A (Used),140000,2026-03-29,Standard
4,Apple iPhone 13 128GB (Used),80000,2026-03-29,128GB


**Create a Dynamic bar chart by taking input from the user**

In [8]:
import re
import plotly.express as px

# 1. User Inputs
user_input = input("Enter your maximum budget (e.g., 170000): ")
top_n_input = input("How many cheapest results do you want? (e.g., 5): ")

try:
    user_budget = int(user_input.replace(",", ""))
    top_n = int(top_n_input)

    # Clean data (ensure capacity exists)
    if 'Capacity' not in df.columns:
        def extract_capacity(name):
            match = re.search(r'(\d+)\s*(GB|TB|gb|tb)', str(name))
            return match.group(0).upper() if match else "Standard"
        df['Capacity'] = df['Name'].apply(extract_capacity)

    # 🔥 STEP A: Filter the entire CSV for the budget first
    entire_filtered = df[df['Price'] <= user_budget].copy()

    if entire_filtered.empty:
        print(f"❌ No phones found under Rs. {user_budget:,.2f}")
    else:
        # 🔥 STEP B: Group and find average
        avg_prices = entire_filtered.groupby(['Name', 'Capacity'])['Price'].mean().reset_index()

        # 🔥 STEP C: Sort the whole market list by price (Closest to budget first)
        sorted_market = avg_prices.sort_values(by='Price', ascending=False)

        # 🔥 STEP D: Take only the top N results from the sorted global list
        final_display = sorted_market.head(top_n)

        # Chart Setup
        calculated_height = max(400, (len(final_display) * 30) + 100)
        
        fig1 = px.bar(
            final_display, x='Price', y='Name', color='Capacity', orientation='h',
            title=f"💰 Absolute Top {top_n} Deals Under Rs. {user_budget:,.2f}",
            labels={'Price': 'Average Price (Rs.)', 'Name': 'Phone Model'},
            color_discrete_sequence=px.colors.qualitative.Pastel,
            height=calculated_height
        )

        fig1.update_layout(template="plotly_dark", yaxis={'categoryorder': 'total descending'})
        fig1.show()
        
except ValueError:
    print("❌ Invalid input! Please enter numbers only.")

**Price Trend Analysis (Line Chart)**

In [4]:
# 1.Ask the user for a specific phone model to track
search_phone = input("Enter a phone model to track price trends (e.g., 'iPhone 13 Pro Max'): ")

# 2. Filter data for that specific phone (Case-insensitive search)
phone_data = df[df['Name'].str.contains(search_phone, case=False, na=False)]

if phone_data.empty:
    print(f"❌ No data found for  '{search_phone}'. Please check the spelling or try a different model.")
else:
    # 3. Group by Date & Capacity and get the average price for each day
    trend_data = phone_data.groupby(['Date', 'Capacity'])['Price'].mean().reset_index()

    # 4. Generate the Interactive Line Chart
    fig2 = px.line(
        trend_data,
        x='Date',
        y='Price',
        color='Capacity',
        title=f"📈 Price Trend Analysis for '{search_phone.upper()}'",
        labels={'Price': 'Average Price (Rs.)', 'Date': 'Date Scraped'},
        markers=True # Adds dots to each data point for better visibility
    )
    
    # 5. Styling the line chart
    fig2.update_layout(
        template="plotly_dark",
        hovermode="x unified" # Shows all capacity prices together when hovering on a date
    )
    
    fig2.show()